In [37]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import warnings

warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    torch.cuda.empty_cache()

BASE_DIR  = "SCVD_converted"
TRAIN_DIR = os.path.join(BASE_DIR, "Train")
TEST_DIR  = os.path.join(BASE_DIR, "Test")

CLASSES     = ["Normal", "Violence", "Weaponized"]
NUM_CLASSES = len(CLASSES)

Using device: cuda
GPU Name: NVIDIA GeForce RTX 4060 Laptop GPU


In [ ]:
class SlowFastDataset(Dataset):
    def __init__(self, data_dir, classes, num_frames=32, resize=(224, 224), alpha=4):
        self.data_dir   = data_dir
        self.classes    = classes
        self.num_frames = num_frames
        self.resize     = resize
        self.alpha      = alpha

        self.video_paths = []
        self.labels      = []

        for label_idx, class_name in enumerate(self.classes):
            class_dir = os.path.join(self.data_dir, class_name)
            if not os.path.exists(class_dir):
                continue
            for file in os.listdir(class_dir):
                if file.lower().endswith('.avi'):
                    self.video_paths.append(os.path.join(class_dir, file))
                    self.labels.append(label_idx)

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):
        vid_path = self.video_paths[idx]
        label    = self.labels[idx]

        cap          = cv2.VideoCapture(vid_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        frames       = []

        if total_frames > 0:
            sample_indices = set(
                np.linspace(0, total_frames - 1, self.num_frames, dtype=int).tolist()
            )
        else:
            sample_indices = set([0] * self.num_frames)

        current_frame = 0
        while cap.isOpened() and len(frames) < self.num_frames:
            ret, frame = cap.read()
            if not ret:
                break
            if current_frame in sample_indices:
                frame = cv2.resize(frame, self.resize)
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frame = frame.astype(np.float32) / 255.0
                frames.append(frame)
            current_frame += 1
        cap.release()

        # Pad if video was too short
        while len(frames) < self.num_frames:
            frames.append(np.zeros((*self.resize, 3), dtype=np.float32))

        video_tensor = torch.tensor(np.array(frames)).permute(3, 0, 1, 2)

        mean = torch.tensor([0.45, 0.45, 0.45]).view(3, 1, 1, 1)
        std  = torch.tensor([0.225, 0.225, 0.225]).view(3, 1, 1, 1)
        video_tensor = (video_tensor - mean) / std

        fast_pathway  = video_tensor
        slow_indices  = torch.linspace(
            0, video_tensor.shape[1] - 1,
            video_tensor.shape[1] // self.alpha
        ).long()
        slow_pathway = torch.index_select(video_tensor, 1, slow_indices)

        return [slow_pathway, fast_pathway], label

In [39]:
train_dataset = SlowFastDataset(TRAIN_DIR, CLASSES)
test_dataset  = SlowFastDataset(TEST_DIR,  CLASSES)

BATCH_SIZE  = 2
NUM_WORKERS = 0

# ✅ Weighted sampler to balance Normal(200) vs Violence(99) vs Weaponized(100)
class_counts   = np.bincount(train_dataset.labels)
print(f"Train class distribution: {dict(zip(CLASSES, class_counts))}")

class_weights  = 1.0 / class_counts
sample_weights = [class_weights[label] for label in train_dataset.labels]
sampler = torch.utils.data.WeightedRandomSampler(
    weights     = sample_weights,
    num_samples = len(sample_weights),
    replacement = True
)

# ✅ Use sampler instead of shuffle=True
train_loader = DataLoader(
    train_dataset,
    batch_size  = BATCH_SIZE,
    sampler     = sampler,
    num_workers = NUM_WORKERS,
    pin_memory  = True
)
test_loader = DataLoader(
    test_dataset,
    batch_size  = BATCH_SIZE,
    shuffle     = False,
    num_workers = NUM_WORKERS,
    pin_memory  = True
)

print(f"Total training videos : {len(train_dataset)}")
print(f"Total testing videos  : {len(test_dataset)}")

Train class distribution: {'Normal': np.int64(200), 'Violence': np.int64(99), 'Weaponized': np.int64(100)}
Total training videos : 399
Total testing videos  : 82


In [40]:
# Check test set distribution
for class_name in CLASSES:
    class_dir = os.path.join(TEST_DIR, class_name)
    count = len([f for f in os.listdir(class_dir) if f.lower().endswith('.avi')])
    print(f"Test {class_name}: {count} videos ({100*count/82:.1f}%)")

Test Normal: 46 videos (56.1%)
Test Violence: 12 videos (14.6%)
Test Weaponized: 24 videos (29.3%)


In [41]:
# Paste this AFTER model = FineTunedSlowFast(NUM_CLASSES).to(device)
# to verify the model's output shape is correct

model.eval()
with torch.no_grad():
    dummy_fast = torch.randn(1, 3, 32, 224, 224).to(device)
    dummy_slow = torch.randn(1, 3, 8, 224, 224).to(device)
    try:
        out = model([dummy_slow, dummy_fast])
        print(f"✅ Model output shape: {out.shape}")  # Should be [1, 3]
        print(f"✅ Output looks correct!" if out.shape == torch.Size([1, 3]) else "❌ Wrong output shape!")
    except Exception as e:
        print(f"❌ Forward pass failed: {e}")

# Also print which layers are trainable
print("\nTrainable layers:")
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"  ✅ {name} — {param.shape}")

✅ Model output shape: torch.Size([1, 3])
✅ Output looks correct!

Trainable layers:
  ✅ base_model.blocks.3.multipathway_blocks.0.res_blocks.4.branch2.conv_a.weight — torch.Size([256, 1024, 3, 1, 1])
  ✅ base_model.blocks.3.multipathway_blocks.0.res_blocks.4.branch2.norm_a.weight — torch.Size([256])
  ✅ base_model.blocks.3.multipathway_blocks.0.res_blocks.4.branch2.norm_a.bias — torch.Size([256])
  ✅ base_model.blocks.3.multipathway_blocks.0.res_blocks.4.branch2.conv_b.weight — torch.Size([256, 256, 1, 3, 3])
  ✅ base_model.blocks.3.multipathway_blocks.0.res_blocks.4.branch2.norm_b.weight — torch.Size([256])
  ✅ base_model.blocks.3.multipathway_blocks.0.res_blocks.4.branch2.norm_b.bias — torch.Size([256])
  ✅ base_model.blocks.3.multipathway_blocks.0.res_blocks.4.branch2.conv_c.weight — torch.Size([1024, 256, 1, 1, 1])
  ✅ base_model.blocks.3.multipathway_blocks.0.res_blocks.4.branch2.norm_c.weight — torch.Size([1024])
  ✅ base_model.blocks.3.multipathway_blocks.0.res_blocks.4.branch2.

In [44]:
# ✅ Separate params cleanly with exact matching to avoid overlap
blocks3_params = []
blocks4_params = []
blocks56_params = []

for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    # Use exact segment matching to avoid overlap
    parts = name.split('.')
    # Find which top-level block this param belongs to
    if 'blocks' in parts:
        block_idx = parts[parts.index('blocks') + 1]
        if block_idx == '3':
            blocks3_params.append(param)
        elif block_idx == '4':
            blocks4_params.append(param)
        elif block_idx in ('5', '6'):
            blocks56_params.append(param)

print(f"blocks[3] trainable tensors : {len(blocks3_params)}")
print(f"blocks[4] trainable tensors : {len(blocks4_params)}")
print(f"blocks[5+6] trainable tensors: {len(blocks56_params)}")

optimizer = optim.AdamW([
    {'params': blocks3_params,  'lr': 5e-6,  'weight_decay': 1e-2},
    {'params': blocks4_params,  'lr': 1e-5,  'weight_decay': 1e-2},
    {'params': blocks56_params, 'lr': 5e-5,  'weight_decay': 1e-2},
])

blocks[3] trainable tensors : 36
blocks[4] trainable tensors : 36
blocks[5+6] trainable tensors: 2


In [45]:
EPOCHS            = 20
accumulation_steps = 4
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
best_val_acc = 0.0

for epoch in range(EPOCHS):
    # --- Training ---
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    optimizer.zero_grad()

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
    for i, (inputs, labels) in enumerate(loop):
        inputs = [inp.to(device, non_blocking=True) for inp in inputs]
        labels = labels.to(device, non_blocking=True)

        outputs = model(inputs)
        loss    = criterion(outputs, labels) / accumulation_steps
        loss.backward()

        if (i + 1) % accumulation_steps == 0 or (i + 1) == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()

        running_loss += loss.item() * accumulation_steps
        _, predicted  = outputs.max(1)
        total        += labels.size(0)
        correct      += predicted.eq(labels).sum().item()
        loop.set_postfix(
            loss=f"{loss.item()*accumulation_steps:.3f}",
            acc =f"{100.*correct/total:.1f}%"
        )

    print(f"Epoch [{epoch+1}/{EPOCHS}] Train Loss: {running_loss/len(train_loader):.4f} | Train Acc: {100.*correct/total:.2f}%")

    # --- Validation with per-class breakdown ---
    model.eval()
    val_loss      = 0.0
    class_correct = [0] * NUM_CLASSES
    class_total   = [0] * NUM_CLASSES

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = [inp.to(device, non_blocking=True) for inp in inputs]
            labels = labels.to(device, non_blocking=True)

            outputs  = model(inputs)
            val_loss += criterion(outputs, labels).item()

            _, predicted = outputs.max(1)
            for c in range(NUM_CLASSES):
                mask = labels == c
                class_correct[c] += predicted[mask].eq(labels[mask]).sum().item()
                class_total[c]   += mask.sum().item()

    val_acc        = 100. * sum(class_correct) / sum(class_total)
    epoch_val_loss = val_loss / len(test_loader)

    print(f"Epoch [{epoch+1}/{EPOCHS}] Test  Loss: {epoch_val_loss:.4f} | Test  Acc: {val_acc:.2f}%")

    # ✅ Per-class accuracy — key indicator of real learning
    for c in range(NUM_CLASSES):
        if class_total[c] > 0:
            print(f"   {CLASSES[c]:12s}: {class_correct[c]}/{class_total[c]} = "
                  f"{100.*class_correct[c]/class_total[c]:.1f}%")

    scheduler.step()

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "slowfast_best.pth")
        print(f"  ✅ Best model saved at epoch {epoch+1} ({val_acc:.2f}%)\n")
    else:
        print()

print(f"Training Complete! Best Test Acc: {best_val_acc:.2f}%")

Epoch 1/20 [Train]: 100%|██████████| 200/200 [02:33<00:00,  1.30it/s, acc=33.8%, loss=1.390]


Epoch [1/20] Train Loss: 1.1377 | Train Acc: 33.83%
Epoch [1/20] Test  Loss: 1.1747 | Test  Acc: 20.73%
   Normal      : 0/46 = 0.0%
   Violence    : 12/12 = 100.0%
   Weaponized  : 5/24 = 20.8%
  ✅ Best model saved at epoch 1 (20.73%)



Epoch 2/20 [Train]: 100%|██████████| 200/200 [02:25<00:00,  1.37it/s, acc=40.1%, loss=1.234]


Epoch [2/20] Train Loss: 1.0743 | Train Acc: 40.10%
Epoch [2/20] Test  Loss: 1.1682 | Test  Acc: 28.05%
   Normal      : 0/46 = 0.0%
   Violence    : 6/12 = 50.0%
   Weaponized  : 17/24 = 70.8%
  ✅ Best model saved at epoch 2 (28.05%)



Epoch 3/20 [Train]: 100%|██████████| 200/200 [02:33<00:00,  1.30it/s, acc=44.4%, loss=1.408]


...

Epoch [18/20] Train Loss: 0.6992 | Train Acc: 82.46%
Epoch [18/20] Test  Loss: 0.9269 | Test  Acc: 51.22%
   Normal      : 17/46 = 37.0%
   Violence    : 7/12 = 58.3%
   Weaponized  : 18/24 = 75.0%



Epoch 19/20 [Train]: 100%|██████████| 200/200 [02:28<00:00,  1.34it/s, acc=83.7%, loss=1.053]


Epoch [19/20] Train Loss: 0.7108 | Train Acc: 83.71%
Epoch [19/20] Test  Loss: 0.9090 | Test  Acc: 53.66%
   Normal      : 17/46 = 37.0%
   Violence    : 7/12 = 58.3%
   Weaponized  : 20/24 = 83.3%



Epoch 20/20 [Train]: 100%|██████████| 200/200 [02:30<00:00,  1.33it/s, acc=83.7%, loss=1.238]


Epoch [20/20] Train Loss: 0.7018 | Train Acc: 83.71%
Epoch [20/20] Test  Loss: 0.8997 | Test  Acc: 53.66%
   Normal      : 17/46 = 37.0%
   Violence    : 7/12 = 58.3%
   Weaponized  : 20/24 = 83.3%

Training Complete! Best Test Acc: 53.66%


In [ ]:
from ultralytics import YOLO
import os
import pandas as pd

# 1) Load YOLO checkpoint
model = YOLO("yolov9e.pt")

# 2) Train
train_results = model.train(
    data="X:/Epics/Acidental/data.yaml",
    epochs=25,
    imgsz=640,
    batch=6,
    workers=8,
    device=0,
    project="X:/Epics/Acidental/runs",
    name="yolov9e_accident",
    exist_ok=True,
    val=True,
    plots=True,
    save=True,
    patience=20,
 )

# 3) Validate and collect accuracy-style metrics
try:
    val_metrics = model.val(data="X:/Epics/Acidental/data.yaml", split="val", imgsz=640, batch=6, device=0)
except Exception:
    # Fallback if your data.yaml defines only test split
    val_metrics = model.val(data="X:/Epics/Acidental/data.yaml", split="test", imgsz=640, batch=6, device=0)

precision = float(val_metrics.box.mp)
recall = float(val_metrics.box.mr)
map50 = float(val_metrics.box.map50)
map50_95 = float(val_metrics.box.map)
f1 = (2 * precision * recall) / (precision + recall + 1e-9)

print("\n================ YOLO METRICS ================")
print(f"Precision (P)      : {precision * 100:.2f}%")
print(f"Recall (R)         : {recall * 100:.2f}%")
print(f"mAP@50             : {map50 * 100:.2f}%")
print(f"mAP@50-95          : {map50_95 * 100:.2f}%")
print(f"F1 score           : {f1 * 100:.2f}%")
print("Accuracy to report : mAP@50 (simple) and mAP@50-95 (strict).")
print("=============================================\n")

# 4) Optional: Read final epoch metrics from results.csv
save_dir = getattr(train_results, "save_dir", None)
if save_dir:
    csv_path = os.path.join(str(save_dir), "results.csv")
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        last = df.iloc[-1]
        print("Last epoch summary from results.csv:")
        for col in [
            "metrics/precision(B)",
            "metrics/recall(B)",
            "metrics/mAP50(B)",
            "metrics/mAP50-95(B)",
        ]:
            if col in df.columns:
                print(f"  {col}: {last[col] * 100:.2f}%")
    else:
        print(f"results.csv not found at: {csv_path}")